In [1]:
# Cài đặt các thư viện cần thiết (dùng bản gpu cho FAISS để tăng tốc nếu Colab bật T4 GPU)
!pip install -q faiss-cpu sentence-transformers pyvi rank_bm25

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 40.1 MB/s eta 0:00:00


In [2]:
import os
import json
import pickle
import numpy as np
import faiss
import torch
from pyvi import ViTokenizer
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from google.colab import drive
print("🔄 Đang kết nối với Google Drive...")
drive.mount('/content/drive')

DATA_PATH = '/content/drive/MyDrive/vimedaq-project/data/raw/vimedaq_full.json'
CORPUS_OUT = '/content/drive/MyDrive/vimedaq-project/data/processed/medical_corpus_V3.json'
FAISS_OUT = '/content/drive/MyDrive/vimedaq-project/data/processed/faiss_index_V3.bin'
BM25_OUT = '/content/drive/MyDrive/vimedaq-project/data/processed/bm25_index_V3.pkl'
EMBEDDER_MODEL = "keepitreal/vietnamese-sbert"

def build_retrieval_system():
    # BƯỚC 1: ĐỌC DỮ LIỆU & TẠO MEDICAL CORPUS
    print("⏳ [1/3] Đang đọc dữ liệu và xây dựng Medical Corpus...")
    with open(DATA_PATH, "r", encoding="utf-8") as f:
        raw_data = json.load(f)

    corpus_texts = []
    metadata = []

    # HASH O(1) ĐỂ LỌC TRÙNG
    seen_contexts = set()
    duplicate_count = 0

    for item in raw_data:
        # Xử lý linh hoạt các key phổ biến
        context = item.get("context") or item.get("content") or item.get("text", "")
        context = context.strip()

        # Bỏ qua nếu context rỗng
        if not context:
            continue

        # LỌC TRÙNG LẶP O(1)
        if context in seen_contexts:
            duplicate_count += 1
            continue

        # Thêm vào kho băm để đánh dấu đã xử lý
        seen_contexts.add(context)

        title = item.get("title", "").strip()
        keyword = item.get("keyword", "").strip()

        # Tạo đoạn text tổng hợp giàu ngữ nghĩa để index
        doc_text = ""
        if keyword: doc_text += f"[{keyword}] "
        if title: doc_text += f"{title}. " # Nên dùng dấu chấm để ngắt câu tự nhiên hơn dấu hai chấm
        doc_text += context

        corpus_texts.append(doc_text.strip())

        # Lưu lại metadata để sau này ViT5 dùng
        metadata.append({
            "question_idx": item.get("question_idx", ""),
            "keyword": keyword,
            "title": title,
            "context": context # Giữ lại context gốc nguyên bản
        })

    # Lưu Corpus và Metadata
    with open(CORPUS_OUT, "w", encoding="utf-8") as f:
        json.dump({"texts": corpus_texts, "metadata": metadata}, f, ensure_ascii=False, indent=4)

    print(f"   Đã tạo corpus với {len(corpus_texts)} tài liệu.")
    print(f"  Đã loại bỏ nhanh {duplicate_count} đoạn văn bản trùng lặp.")

    # BƯỚC 2: XÂY DỰNG BM25 INDEX (TÌM KIẾM TỪ KHÓA)
    print("\n⏳ [2/3] Đang xây dựng BM25 Index...")
    # Tokenize tiếng Việt trước khi đưa vào BM25
    tokenized_corpus = [ViTokenizer.tokenize(text).lower().split() for text in corpus_texts]
    bm25 = BM25Okapi(tokenized_corpus)

    with open(BM25_OUT, "wb") as f:
        pickle.dump(bm25, f)
    print("   Đã lưu BM25 Index.")

    # BƯỚC 3: XÂY DỰNG FAISS INDEX (TÌM KIẾM NGỮ NGHĨA)
    print("\n⏳ [3/3] Đang xây dựng FAISS Index...")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = SentenceTransformer(EMBEDDER_MODEL, device=device)

    # Encode toàn bộ corpus sang vector
    embeddings = model.encode(corpus_texts, show_progress_bar=True, normalize_embeddings=True)
    embeddings = np.array(embeddings).astype('float32')

    # Dùng Inner Product (chạy tương đương Cosine Similarity vì data đã normalize)
    dimension = embeddings.shape[1]
    faiss_index = faiss.IndexFlatIP(dimension)
    faiss_index.add(embeddings)

    faiss.write_index(faiss_index, FAISS_OUT)
    print("    Đã lưu FAISS Index.")
    print("\n HOÀN TẤT XÂY DỰNG HỆ THỐNG RETRIEVAL!")


if __name__ == "__main__":

     build_retrieval_system()


Đang kết nối với Google Drive...
Mounted at /content/drive
 [1/3] Đang đọc dữ liệu và xây dựng Medical Corpus...
   Đã tạo corpus với 17955 tài liệu.
   Đã loại bỏ nhanh 26273 đoạn văn bản trùng lặp.

 [2/3] Đang xây dựng BM25 Index...
    Đã lưu BM25 Index.

[3/3] Đang xây dựng FAISS Index...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.90k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/895k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.14M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/562 [00:00<?, ?it/s]

    Đã lưu FAISS Index.

 HOÀN TẤT XÂY DỰNG HỆ THỐNG RETRIEVAL!
